In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw")

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [4]:
pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.6/983.6 kB 7.9 MB/s  0:00:00

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import torch
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

In [8]:
deberta_name = "microsoft/deberta-v3-small"
roberta_name = "roberta-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_name)
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_name)

deberta = AutoModelForSequenceClassification.from_pretrained(
    deberta_name,
    num_labels=5,
    ignore_mismatched_sizes=True
)

roberta = AutoModelForSequenceClassification.from_pretrained(
    roberta_name,
    num_labels=5,
    ignore_mismatched_sizes=True
)

deberta.eval()
roberta.eval()

[transformers] Could not extract SentencePiece model from /Users/nishantskuttan/.cache/huggingface/hub/models--microsoft--deberta-v3-small/snapshots/a36c739020e01763fe789b4b85e2df55d6180012/spm.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


ValueError: Error parsing line b'\x0e' in /Users/nishantskuttan/.cache/huggingface/hub/models--microsoft--deberta-v3-small/snapshots/a36c739020e01763fe789b4b85e2df55d6180012/spm.model

In [ ]:
def build_prompt(row):

    text = f"""
Question:
{row['prompt']}

A. {row['A']}

B. {row['B']}

C. {row['C']}

D. {row['D']}

E. {row['E']}
"""

    return text.strip()

In [ ]:
def predict(model, tokenizer, text):

    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = F.softmax(logits, dim=-1)

    return probs.squeeze().numpy()

In [ ]:
sample = build_prompt(train.iloc[25])

deberta_probs = predict(
    deberta,
    deberta_tokenizer,
    sample
)

labels = ["A","B","C","D","E"]

answer = labels[deberta_probs.argmax()]
prob = deberta_probs.max()

print(answer, prob)

In [ ]:
roberta_probs = predict(
    roberta,
    roberta_tokenizer,
    sample
)

ensemble = (deberta_probs + roberta_probs)/2

answer = labels[ensemble.argmax()]

print(answer)

In [ ]:
ensemble = (
    0.70*deberta_probs
    +
    0.30*roberta_probs
)

answer = labels[ensemble.argmax()]

print(answer)

In [ ]:
top3 = ensemble.argsort()[::-1][:3]

prediction = " ".join(
    labels[i] for i in top3
)

print(prediction)

In [ ]:
predictions = []

for _,row in test.iterrows():

    text = build_prompt(row)

    d = predict(
        deberta,
        deberta_tokenizer,
        text
    )

    r = predict(
        roberta,
        roberta_tokenizer,
        text
    )

    probs = 0.7*d + 0.3*r

    top3 = probs.argsort()[::-1][:3]

    pred = " ".join(
        labels[i]
        for i in top3
    )

    predictions.append(pred)

In [ ]:
changed = 0

for i in range(50):

    row = test.iloc[i]

    # Original prompt
    text1 = build_prompt(row)

    # TTA prompt
    text2 = (
        "Answer the following multiple-choice question carefully:\n\n"
        + text1
    )

    # First inference
    p1 = predict(
        deberta,
        deberta_tokenizer,
        text1
    )

    # Second inference
    p2 = predict(
        deberta,
        deberta_tokenizer,
        text2
    )

    # Average probabilities
    final_probs = (p1 + p2) / 2

    # Top-1 labels
    original = p1.argmax()
    augmented = final_probs.argmax()

    if original != augmented:
        changed += 1

print(changed)

In [ ]:
changed = 0

for i in range(100):

    row = test.iloc[i]
    text = build_prompt(row)

    # DeBERTa probabilities
    d_probs = predict(
        deberta,
        deberta_tokenizer,
        text
    )

    # RoBERTa probabilities
    r_probs = predict(
        roberta,
        roberta_tokenizer,
        text
    )

    # Weighted ensemble
    ensemble_probs = 0.7 * d_probs + 0.3 * r_probs

    # Compare Top-1 predictions
    deberta_pred = d_probs.argmax()
    ensemble_pred = ensemble_probs.argmax()

    if deberta_pred != ensemble_pred:
        changed += 1

print(changed)

In [ ]:
positive_gain = 0

for i in range(100):

    row = test.iloc[i]
    text = build_prompt(row)

    # DeBERTa
    d_probs = predict(
        deberta,
        deberta_tokenizer,
        text
    )

    # RoBERTa
    r_probs = predict(
        roberta,
        roberta_tokenizer,
        text
    )

    # Weighted Ensemble
    ensemble_probs = (
        0.7 * d_probs +
        0.3 * r_probs
    )

    # Highest confidence
    deberta_conf = d_probs.max()
    ensemble_conf = ensemble_probs.max()

    # Confidence gain
    gain = ensemble_conf - deberta_conf

    if gain > 0:
        positive_gain += 1

print(positive_gain)

In [ ]:
labels = ["A", "B", "C", "D", "E"]

predictions = []

for i in range(100):

    row = test.iloc[i]      # validation dataframe

    text = build_prompt(row)

    d_probs = predict(deberta, deberta_tokenizer, text)
    r_probs = predict(roberta, roberta_tokenizer, text)

    ensemble_probs = (
        0.7 * d_probs +
        0.3 * r_probs
    )

    top3 = ensemble_probs.argsort()[::-1][:3]

    predictions.append(
        [labels[j] for j in top3]
    )

In [ ]:
actual = test["answer"].tolist()[:100]

In [ ]:
def apk(actual, predicted, k=3):

    if actual in predicted[:k]:
        rank = predicted.index(actual) + 1
        return 1.0 / rank

    return 0.0

In [ ]:
import numpy as np

map3 = np.mean([
    apk(a, p)
    for a, p in zip(actual, predictions)
])

print(round(map3, 4))